# 082 — Dimensionar hardware: de la laptop al clúster

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

- **Ecuación de memoria**: `M_total = pesos + KV cache + activaciones +
  overhead`, más un 15–20 % de margen. Sólo el primer sumando es el que se suele
  calcular.
- **Pesos** = `N × bytes_por_parámetro`: 2 B en FP16, 1 B en FP8/INT8, ≈ 0,57 B
  en GGUF Q4_K_M. Un 8B va de 32 GB (FP32) a 4,6 GB (Q4).
- **KV cache** = `2 · capas · kv_heads · d_head · seq · lote · bytes`. Con
  concurrencia y contexto largo puede superar a los pesos.
- **Entrenar es otra escala**: fine-tuning completo con Adam ≈ 16 B/parámetro
  (un 8B pide ~128 GB); QLoRA lo baja a ~6–8 GB.
- **Tres escalones**: portátil/unificada (3B–14B en Q4), estación de 1–2 GPUs
  (30B–70B en Q4, QLoRA), nodo con NVLink (BF16 completo y entrenamiento).


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("evaluation", seed=82)
show(result)


## Reflexión

1. En el ejercicio 2 el KV cache triplica a los pesos. ¿Qué tres palancas tienes
   para que quepa, y cuál degrada menos el producto?
2. Tu equipo pide "una GPU más grande" porque el modelo no entra. ¿Qué preguntas
   harías antes de aprobar la compra?
3. ¿En qué condiciones una máquina con memoria unificada de 192 GB es mejor
   elección que dos GPUs de 48 GB, y en cuáles es claramente peor?
